In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so

from formulaic import Formula

# Herramientas de modelos lineales
from sklearn import linear_model  

# Medidas de desempeño
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error    

# Regresión Polinomial

### Laboratorio de Datos, IC - FCEN - UBA - 1C 2026

Buscamos los coeficientes de un polinomio de grado $n$
$$Y = \beta_0 + \beta_1 X + \beta_2 X^2 + \beta_3 X^3 + \dots + \beta_n X^n$$

que mejor aproxime a nuestros datos en el sentido de Cuadrados Mínimos. Es decir, buscamos $\beta_1,\dots, \beta_n$ que minimicen:
$$ RSS(\beta) = \displaystyle\sum_{i=1}^n (y_i - P(x_i))^2 $$

Vamos a usar como ejemplo los datos de PBI per capita de Argentina, del dataset `gapminder`

In [ ]:
from gapminder import gapminder
datos_arg = gapminder[gapminder['country'] == 'Argentina'].reset_index(drop=True)
datos_arg

### Visualización

Teníamos la visualización de la Regresión Lineal:

In [ ]:
(
    so.Plot(data=datos_arg, x='year', y='gdpPercap')
    .add(so.Dot())
    .add(so.Line(color='r', linewidth=3), so.PolyFit(1))
)

Calculemos el $R^2$ de la Regresión Lineal:

In [ ]:
# Escribimos la fórmula
y, X = Formula('gdpPercap ~ year').get_model_matrix(datos_arg)

# Inicializamos el modelo de regresión
modelo = linear_model.LinearRegression(fit_intercept=False)

# Realizamos el ajuste
modelo.fit(X, y)

# Calculamos R²
y_pred = modelo.predict(X)
r2_score(datos_arg['gdpPercap'], y_pred)

Para cambiar el grado del polinomio que ajusta los datos en Seaborn, simplemente cambios el argumento de `so.PolyFit`. Por ejemplo, para un polinomio de grado $???$:

In [ ]:
(
    so.Plot(data=datos_arg, x=datos_arg['year'], y='gdpPercap')
    .add(so.Dot())
    .add(so.Line(color='r', linewidth=3), so.PolyFit(???))
)

### Cálculo de coeficientes y de predicciones

Queremos buscar el polinomio de grado 3 que mejor aproxima a los datos en sentido de cuadrados mínimos:
$$P(x) = \beta_0 + \beta_1 x + \beta_2 x^ 2 + \beta_3 x^ 3$$

Calcularemos los coeficientes con `scikit-learn`. Es parecido a lo que hacíamos con la regresión lineal, pero con un paso extra: tenemos que construir la matriz de datos agregando columnas para $x^2$ y $x^3$.

### Formulaic: implementación de alto rendimiento de las fórmulas de Wilkinson para Python.

Aunque perfectamente podemos hacer la matriz "a mano", el paquete Formulaic nos permite hacerla automáticamente para polinomios de cualquier grado.

In [ ]:
# poly(year,3) significa que usamos un polinomio de grado 3 cuya variable es year
# raw = True nos da los datos "crudos", es decir genera columnas x, x^2 y x^3 sin ninguna transformación ni normalización.
# -1 significa que no queremos una columna de 1's en la matriz para el intercept

y, X = Formula('gdpPercap ~ poly(year, 3, raw=True) - 1').get_model_matrix(datos_arg)
y = y.iloc[:, 0]  # Convertimos el DataFrame a Series

In [ ]:
X

In [ ]:
y

In [ ]:
# Inicializamos el modelo de regresión. 
modelo = linear_model.LinearRegression(fit_intercept=True)

# Realiza el ajuste
modelo.fit(X, y)

# Como usamos fit_intercept = True, la ordenada al origen está en modelo.intercept_
ordenada_al_origen = modelo.intercept_

# Recuperamos los valores de los demas coeficientes (de menor potencia a la mayor)
coeficientes = modelo.coef_

# Armamos el array con los valores de beta
beta = np.array([ordenada_al_origen, *coeficientes])

# Imprimimos los valores:
print('beta_0: ', beta[0])
print('beta_1: ', beta[1])
print('beta_2: ', beta[2])
print('beta_3: ', beta[3])

El polinomio de grado a lo sumo 3 que mejor aproxima a los datos es:
$$P(x) = -984937719.58 + 1491410.41 x -752.77 x^ 2 + 0.12 x^ 3 $$

Corroboremos que el gráfico de este polinomio es el mismo que arma `seaborn` con `so.PolyFit`:

In [ ]:
X

In [ ]:
(
    so.Plot(data=datos_arg, x='year', y='gdpPercap')
    .add(so.Dot())
    .add(so.Line(color='green', linewidth=2), y=modelo.predict(X), label='sklearn')
    .add(so.Line(color='red', linestyle='--', linewidth=2), so.PolyFit(3), label='seaborn')
    .label()
)

Igual que antes, podemos estimar el PBI per capita, por ejemplo para 1990 y 2015:

* haciendo las cuentas

In [ ]:
# Haciendo las cuentas a mano
prediccion_1990 = ???

# O usando numpy
prediccion_2015 = beta @ 2015**np.arange(4, dtype='int64')

print('Prediccion 1990: ', prediccion_1990)
print('Prediccion 2015: ', prediccion_2015)

* usando `predict`

In [ ]:
# Armamos un DataFrame con los datos de x a los cuales les queremos aplicar el modelo
nuevos = pd.DataFrame({"year":  [1990, 2015]})

# Armamos el DataFrame para pasarle a predict
datos_a_predecir = Formula('gdpPercap ~ poly(year, 3, raw=True) - 1').rhs.get_model_matrix(nuevos)   #rhs (right hand side) nos da solo x
datos_a_predecir

In [ ]:
modelo.predict(datos_a_predecir)

También podemos calcular las medidas de desempeño del modelo:

In [ ]:
y_pred = modelo.predict(X)

# Calculando el R^2
r2 = r2_score(datos_arg['gdpPercap'], y_pred)
print('R^2: ', r2)

# Calculando el ECM
ecm = mean_squared_error(datos_arg['gdpPercap'], y_pred)
print('ECM: ', ecm)

# Calculando la RECM
recm = root_mean_squared_error(datos_arg['gdpPercap'], y_pred)
print('RECM: ', recm)

## Regresión con funciones bases

Vamos a trabajar con mediciones de dióxido de carbono en el aire en el observatorio de Mauna Loa (Hawaii)

In [ ]:
# Cargamos los datos y graficamos
data = pd.read_csv('co2_hawaii.csv')
data

In [ ]:
# Tomamos la fecha como indice
data = data.set_index("fecha")
data

In [ ]:
type(data)

In [ ]:
# Graficamos
data.plot()

Queremos ajustar a los datos una función de la forma
$$
co2 = \beta_0 + \beta_1 * mes + \beta_2 * \sin\left(2 \pi *  \frac{mes}{12}\right)
$$

Construimos una matriz $X$ con una columna para el mes y otra columna para el seno.

In [ ]:
# ¿Cuántos meses hay en los datos?
data.co2.size

In [ ]:
mes = np.arange(521)
datos_transformados = pd.DataFrame(
    {
        "mes": mes,
        "sinx": ???,
        "co2": data["co2"]
    },
    index=data.index
)
datos_transformados

In [ ]:
# Ajustamos el modelo

y, X = Formula('co2 ~ mes + sinx').get_model_matrix(???)
y = y.iloc[:, 0]  # Convertimos el DataFrame a Series

# Inicializamos el modelo de regresión
modeloCO2 = linear_model.LinearRegression(fit_intercept=???)

# Realizamos el ajuste
modeloCO2.fit(X, y)

In [ ]:
# Predecimos
y_pred = modeloCO2.predict(X)

In [ ]:
# Graficamos
(
    so.Plot()  # Podemos usar también x = data.index pero no se ve bien
    .add(so.Line(), data=data, x = mes, y="co2")
    .add(so.Line(color = "red"), x=mes, y=y_pred, label='Ajuste')
)

In [ ]:
# Calculando el R^2
r2 = r2_score(data["co2"], y_pred)
print('R^2: ', r2)

## Residuos
El $R^2$ nos dio muy bien! Se podrá mejorar?
Veamos los residuos...

In [ ]:
residuos = data["co2"] - y_pred
(
    so.Plot(x=mes, y=residuos)
    .add(so.Dot())
)

Los residuos no parecen al azar... ¡podemos mejorar el ajuste!

## Ejercicio

Agregar otras funciones base al modelo y ver si mejora el ajuste en el gráfico y el $R^2$.

Recordar que solo podemos usar una combinación lineal de funciones base.